# 🎵 KKBox Churn Prediction — Feature Engineering v7
## Scala-Aligned · Vectorized · Bryan-Style Split · No Leakage

---

## 📌 Mục tiêu của notebook này

Notebook này thực hiện **toàn bộ Feature Engineering** cho bài toán dự đoán churn của KKBox,
bao gồm: xây dựng population, gán nhãn churn, tính features, và xuất dataset cho modeling.

Thiết kế được căn chỉnh theo logic của `WSDMChurnLabeller.scala` và
**Bryan Gregory's winning solution** (WSDM KKBox 2018).

---

## 🏗️ Kiến trúc tổng thể

```
Preprocessing notebook
        ↓
clean_transactions_core.parquet  (transactions v1 + v2 đã gộp)
clean_members_core.parquet
        ↓
Feature Engineering notebook (notebook này)
        ↓
  ┌─────────────────────────────────────────┐
  │  Với mỗi tháng target:                  │
  │  1. Xác định population (effective state)│
  │  2. Gán nhãn churn (30-day rule)         │
  │  3. Tính features (lịch sử trước cutoff) │
  │  4. Gộp thành snapshot tháng đó          │
  └─────────────────────────────────────────┘
        ↓
master_model_table.parquet  →  Modeling notebook
inference_snapshot.parquet  →  Submission
```

---

## 📅 Phân chia tập dữ liệu (Bryan-style)

| Tập | Tháng | Mục đích |
|-----|-------|----------|
| **Train** | 2017-01 (January) | Huấn luyện mô hình |
| **Validation** | 2017-02 (February) | Đánh giá, chọn hyperparameter |
| **Inference** | 2017-03 (March) | Dự đoán submission |

### Tại sao chỉ dùng Jan/Feb 2017 thay vì toàn bộ 2015-2016?

Đây là điểm cốt lõi của Bryan's approach:

| Population | Churn rate | Lý do |
|-----------|-----------|-------|
| Users expiry 2015-2016 | ~99% | Là users **đã không active từ lâu** — họ đã churn thật sự |
| Users expiry Jan 2017 | ~6-15% | Là **active subscribers** sắp hết hạn, nhiều người sẽ gia hạn |
| Users expiry Feb 2017 | ~26% | Gần giống Jan, có thể predict được |
| Users expiry Mar 2017 | ? | **Target cần predict** |

> **Nguyên tắc**: Train phải có cùng distribution với target.
> Jan 2017 population ≈ Feb 2017 ≈ Mar 2017 (tất cả là active subscribers sắp hết hạn).
> Training trên 2015-2016 dạy model sai prior: "gần như tất cả đều churn".

---

## 📐 Định nghĩa churn

```
is_churn = 1  nếu user KHÔNG gia hạn trong vòng 30 ngày sau khi membership hết hạn
is_churn = 0  nếu user gia hạn trong vòng 30 ngày sau khi membership hết hạn
```

- Nhãn dựa trên **hành vi gia hạn thực tế**, không phải hành vi nghe nhạc
- User vẫn còn membership dài hạn nhưng không nghe nhạc → **KHÔNG** đổi nhãn

---

## 🔒 Nguyên tắc chống leakage

| Thành phần | Dữ liệu được dùng | Dữ liệu bị cấm |
|-----------|-------------------|----------------|
| **Population** | Transactions ≤ cutoff | Transactions sau cutoff |
| **Label** | Transactions sau last_expire | — |
| **Features** | Mọi activity ≤ cutoff | Mọi activity sau cutoff |

---

## 📦 Input / Output

**Input** (từ Preprocessing notebook):
- `Data/clean_transactions_core.parquet`
- `Data/clean_members_core.parquet`
- `Data/clean_user_logs_hist.parquet` *(auxiliary)*
- `Data/clean_user_logs_march.parquet` *(auxiliary)*
- `Data/preprocessing_metadata_v3.json`

**Output** (cho Modeling notebook):
- `Data/master_model_table.parquet` — train + validation
- `Data/inference_snapshot.parquet` — March 2017 để predict
- `Data/snapshot_summary.csv` — thống kê churn rate theo tháng
- `Data/feature_engineering_metadata_v7.json` — metadata của run


In [66]:

# ===== 1. Imports =====
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import gc

pd.set_option("display.max_columns", 200)

pd.set_option("display.max_rows", 100)

In [67]:

# ===== 2. Paths =====
DATA_DIR = Path("Data")

TX_PATH        = DATA_DIR / "clean_transactions_core.parquet"
MEM_PATH       = DATA_DIR / "clean_members_core.parquet"
LOG_HIST_PATH  = DATA_DIR / "clean_user_logs_hist.parquet"
LOG_MARCH_PATH = DATA_DIR / "clean_user_logs_march.parquet"
META_PATH      = DATA_DIR / "preprocessing_metadata_v3.json"

# Official competition label files (ground-truth)
TRAIN_LABELS_PATH   = DATA_DIR / "train.csv"      # Jan 2017 official labels
TRAIN_V2_LABELS_PATH= DATA_DIR / "train_v2.csv"   # Feb 2017 official labels

for p in [TX_PATH, MEM_PATH, LOG_HIST_PATH, LOG_MARCH_PATH, META_PATH]:
    print(f"{p}: {'FOUND' if p.exists() else 'MISSING'}")
print()
for p in [TRAIN_LABELS_PATH, TRAIN_V2_LABELS_PATH]:
    print(f"{p}: {'FOUND ✅' if p.exists() else 'MISSING ⚠️  (official labels not available)'}")


Data\clean_transactions_core.parquet: FOUND
Data\clean_members_core.parquet: FOUND
Data\clean_user_logs_hist.parquet: FOUND
Data\clean_user_logs_march.parquet: FOUND
Data\preprocessing_metadata_v3.json: FOUND

Data\train.csv: FOUND ✅
Data\train_v2.csv: FOUND ✅


In [68]:

# ===== 3. Load clean tables =====
if not TX_PATH.exists() or not MEM_PATH.exists():
    raise FileNotFoundError("Thiếu clean core tables. Hãy chạy Preprocessing_timebased_v4/v3 trước.")

transactions    = pd.read_parquet(TX_PATH)
members         = pd.read_parquet(MEM_PATH)
user_logs_hist  = pd.read_parquet(LOG_HIST_PATH)  if LOG_HIST_PATH.exists()  else None
user_logs_march = pd.read_parquet(LOG_MARCH_PATH) if LOG_MARCH_PATH.exists() else None

meta = {}
if META_PATH.exists():
    with open(META_PATH, "r", encoding="utf-8") as f:
        meta = json.load(f)

# Load official competition labels if available
official_train_labels = None
official_val_labels   = None

if TRAIN_LABELS_PATH.exists():
    official_train_labels = pd.read_csv(TRAIN_LABELS_PATH)
    official_train_labels.columns = [c.lower() for c in official_train_labels.columns]
    official_train_labels["is_churn"] = official_train_labels["is_churn"].astype("int8")
    print(f"official_train_labels (Jan 2017): {official_train_labels.shape}")
    print(f"  churn rate: {official_train_labels['is_churn'].mean():.3f}")
else:
    print("⚠️  train.csv not found — will use FE-derived labels for Jan 2017")

if TRAIN_V2_LABELS_PATH.exists():
    official_val_labels = pd.read_csv(TRAIN_V2_LABELS_PATH)
    official_val_labels.columns = [c.lower() for c in official_val_labels.columns]
    official_val_labels["is_churn"] = official_val_labels["is_churn"].astype("int8")
    print(f"official_val_labels   (Feb 2017): {official_val_labels.shape}")
    print(f"  churn rate: {official_val_labels['is_churn'].mean():.3f}")
else:
    print("⚠️  train_v2.csv not found — will use FE-derived labels for Feb 2017")

print("\ntransactions:", transactions.shape)
print("members     :", members.shape)
print("user_logs_hist :", None if user_logs_hist is None else user_logs_hist.shape)
print("user_logs_march:", None if user_logs_march is None else user_logs_march.shape)

display(transactions.head())
display(members.head())


official_train_labels (Jan 2017): (992931, 2)
  churn rate: 0.064
official_val_labels   (Feb 2017): (970960, 2)
  churn rate: 0.090

transactions: (1978751, 11)
members     : (6769473, 12)
user_logs_hist : (106543, 10)
user_logs_march: (396362, 10)


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel,expire_gap_days,is_extreme_expiry
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,22,395,1599,1599,0,2016-10-23,2018-02-06,0,471,1
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,41,30,99,99,1,2017-03-15,2017-04-15,0,31,0
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,39,30,149,149,1,2017-02-28,2017-04-19,0,50,0
3,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,39,30,149,149,1,2017-03-31,2017-05-19,0,49,0
4,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,41,30,149,149,1,2017-03-26,2017-04-26,0,31,0


,msno,city,bd,gender,registered_via,bd_missing,city_missing,gender_missing,registered_via_missing,registration_year,registration_month,registration_day
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,27.0,Unknown,11,1,0,1,0,2011,9,11
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,27.0,Unknown,7,1,0,1,0,2011,9,14
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,27.0,Unknown,11,1,0,1,0,2011,9,15
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,27.0,Unknown,11,1,0,1,0,2011,9,15
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32.0,female,9,0,0,0,0,2011,9,15


In [69]:

# ===== 4. Setup months =====
# Bryan Gregory's setup: Train=Jan, Val=Feb, Inf=Mar (all 2017)
# Lý do: population Jan/Feb 2017 là active subscribers sắp hết hạn,
# churn rate realistic (~6-15%). Dùng toàn bộ 2015-2016 cho train
# sẽ kéo population là inactive users đã lâu → 99% churn rate giả tạo.
setup_metadata = meta.get("setup_metadata", {})

TRAIN_MONTHS = pd.period_range("2017-01", "2017-01", freq="M")  # Jan 2017 only
VAL_MONTHS   = pd.period_range("2017-02", "2017-02", freq="M")  # Feb 2017 only
INF_MONTH    = pd.Period("2017-03", freq="M")                    # Mar 2017 inference
GRACE_DAYS   = 30

def month_end(period_m):
    return period_m.to_timestamp(how="end").normalize()

inf_cutoff = month_end(INF_MONTH)

print("TRAIN_MONTHS:", TRAIN_MONTHS[0], "->", TRAIN_MONTHS[-1], "| n =", len(TRAIN_MONTHS))
print("VAL_MONTHS  :", VAL_MONTHS[0], "->", VAL_MONTHS[-1], "| n =", len(VAL_MONTHS))
print("INF_MONTH   :", INF_MONTH, "| cutoff =", inf_cutoff)
print("GRACE_DAYS  :", GRACE_DAYS)


TRAIN_MONTHS: 2017-01 -> 2017-01 | n = 1
VAL_MONTHS  : 2017-02 -> 2017-02 | n = 1
INF_MONTH   : 2017-03 | cutoff = 2017-03-31 00:00:00
GRACE_DAYS  : 30


## 5. Vectorized label builder (replaces per-user loop pipeline)

### Key idea
The original per-user Python loop (`select_effective_state_by_cutoff` + `calculate_renewal_gap_scala_like`) runs at ~6ms/user, making 27 months infeasible.

The vectorized replacement uses pandas `groupby().last()` and join-based renewal gap — equivalent logic, ~100x faster:

1. **Effective state**: `sort_values` + `groupby().last()` across all users at once
2. **Population**: filter where effective expiry period == target month
3. **Renewal gap**: join future transactions → `groupby().min()` → vectorized subtract
4. **Labels**: vectorized boolean assignment

Helper functions `_scala_sort_key`, `select_effective_state_by_cutoff`, `calculate_renewal_gap_scala_like`, and `get_population_users_by_expiry_month` are **no longer needed** and are replaced by `build_labels_vectorized`.


In [70]:
# ===== 5A-7. VECTORIZED label builder =====
# Replaces: _scala_sort_key, _normalize_tx_row, _plan_signature,
#            select_effective_state_by_cutoff, calculate_renewal_gap_scala_like,
#            get_population_users_by_expiry_month, build_labels_from_expiry_month

def build_labels_vectorized(
    transactions: pd.DataFrame,
    target_month: pd.Period,
    grace_days: int = 30,
    debug: bool = False,
):
    """
    Fully vectorized label builder. Equivalent to Scala-aligned state selection
    + renewal gap, but uses pandas groupby instead of per-user Python loops.
    ~100x faster than the row-by-row version.

    Returns DataFrame with columns:
        msno, snapshot_date, last_expire, is_churn, label_source
    """
    cutoff_date = target_month.to_timestamp(how="end").normalize()

    # ── Step 1: history = all rows up to cutoff ───────────────────────────
    hist = transactions[transactions["transaction_date"] <= cutoff_date]

    # ── Step 2: effective state per user via vectorized sort + groupby.last ─
    # Sort order mirrors Scala: tx_date ASC, is_cancel ASC (renewals before cancels
    # on same day), membership_expire_date ASC (longer expiry last among renewals)
    # → last row per user = effective state
    hist_sorted = hist.sort_values(
        ["msno", "transaction_date", "is_cancel", "membership_expire_date"],
        ascending=[True, True, True, True],
    )
    effective = hist_sorted.groupby("msno", sort=False).last().reset_index()

    # ── Step 3: population = users whose effective expiry is in target month ─
    effective["expire_period"] = effective["membership_expire_date"].dt.to_period("M")
    population = (
        effective[effective["expire_period"] == target_month]
        [["msno", "membership_expire_date"]]
        .rename(columns={"membership_expire_date": "last_expire"})
        .copy()
    )

    if len(population) == 0:
        if debug:
            print(f"[{target_month}] No population found — returning empty DataFrame")
        return pd.DataFrame(
            columns=["msno", "snapshot_date", "last_expire", "is_churn", "label_source"]
        )

    # ── Step 4: future non-cancel transactions after each user's last_expire ─
    # NOTE: no > cutoff_date pre-filter here — cutoff only constrains FEATURES.
    # For labels, the renewal window starts at last_expire (per user), not month-end.
    # The per-user filter on line below correctly scopes each user's grace window.
    future = transactions[
        transactions["is_cancel"] != 1
    ].copy()
    future = future.merge(population[["msno", "last_expire"]], on="msno", how="inner")
    future = future[future["transaction_date"] > future["last_expire"]]

    # ── Step 5: first renewal date per user → renewal gap ────────────────
    if len(future) > 0:
        first_renewal = (
            future.groupby("msno")["transaction_date"]
            .min()
            .reset_index()
            .rename(columns={"transaction_date": "first_renewal_date"})
        )
        population = population.merge(first_renewal, on="msno", how="left")
        population["renewal_gap"] = (
            population["first_renewal_date"] - population["last_expire"]
        ).dt.days
    else:
        population["first_renewal_date"] = pd.NaT
        population["renewal_gap"] = np.nan

    # ── Step 6: assign labels ─────────────────────────────────────────────
    max_txn_date = transactions["transaction_date"].max()

    population["is_churn"]      = np.int8(1)
    population["label_source"]  = "no_future_data"

    observed_mask = population["renewal_gap"].notna()
    population.loc[observed_mask, "label_source"] = "observed"
    population.loc[
        observed_mask & (population["renewal_gap"] <= grace_days), "is_churn"
    ] = np.int8(0)

    beyond_mask = population["last_expire"] > max_txn_date
    population.loc[beyond_mask, "label_source"] = "beyond_window"
    population.loc[beyond_mask, "is_churn"]     = np.int8(1)

    population["snapshot_date"] = cutoff_date

    result = population[
        ["msno", "snapshot_date", "last_expire", "is_churn", "label_source"]
    ].copy()

    if debug:
        n          = len(result)
        churn_rate = result["is_churn"].mean()
        observed   = (result["label_source"] == "observed").sum()
        print(
            f"[{target_month}] population={n:,}  churn_rate={churn_rate:.3f}  "
            f"observed={observed:,}  no_future={n - observed:,}"
        )

    return result


In [71]:
# (replaced by build_labels_vectorized in cell above)
# This cell is intentionally left empty.


In [72]:
# (replaced by build_labels_vectorized in cell above)
# This cell is intentionally left empty.


## 6. Population + Label builder (vectorized)

Both population selection and label construction are handled in one pass by `build_labels_vectorized`.
No separate population loop is needed.


In [73]:
# (replaced by build_labels_vectorized in cell above)
# This cell is intentionally left empty.


## 7. Label builder

See `build_labels_vectorized` in cell 06. Replaces the per-user loop version.


In [74]:
# (replaced by build_labels_vectorized in cell above)
# This cell is intentionally left empty.


In [75]:
# ===== Pre-group transactions (kept for reference only) =====
# transactions_grouped is no longer needed by build_labels_vectorized.
# build_core_feature_snapshot still takes the full transactions DataFrame directly.
# This cell can be skipped.

# Compute global max transaction date (still useful for reference)
MAX_TXN_DATE_GLOBAL = transactions["transaction_date"].max()
print(f"Global max transaction date: {MAX_TXN_DATE_GLOBAL}")

PIPELINE_START_TIME = time.time()


Global max transaction date: 2017-03-31 00:00:00


In [76]:
# ===== 8. Runtest labels (FAST DEBUG — vectorized) =====
sample_train_month = pd.Period("2016-12", freq="M")
sample_val_month   = pd.Period("2017-01", freq="M")

# Stratified sample — guarantee users with expiries in target months are included
target_periods = [pd.Period(m, freq="M") for m in
                  ["2016-10", "2016-11", "2016-12", "2017-01", "2017-02"]]
target_msnos = transactions[
    transactions["membership_expire_date"].dt.to_period("M").isin(target_periods)
]["msno"].unique()
other_msnos = (
    transactions[~transactions["msno"].isin(target_msnos)]["msno"]
    .drop_duplicates()
    .sample(max(0, 50000 - len(target_msnos)), random_state=42)
)
debug_users = list(target_msnos) + list(other_msnos)
transactions_debug = transactions[transactions["msno"].isin(debug_users)].copy()

print(f"Debug sample: {len(target_msnos):,} target-month users + "
      f"{len(other_msnos):,} filler = {len(debug_users):,} total")
print(f"Debug transactions rows: {len(transactions_debug):,}")

print("\n" + "="*80)
start_train = time.time()
sample_train_labels = build_labels_vectorized(
    transactions_debug, sample_train_month, grace_days=GRACE_DAYS, debug=True
)
train_time = time.time() - start_train

print("="*80)
start_val = time.time()
sample_val_labels = build_labels_vectorized(
    transactions_debug, sample_val_month, grace_days=GRACE_DAYS, debug=True
)
val_time = time.time() - start_val

print("="*80)
print(f"sample_train_labels shape : {sample_train_labels.shape} ({train_time:.2f}s)")
print(f"sample_val_labels shape   : {sample_val_labels.shape} ({val_time:.2f}s)")
print(f"\nLabel source breakdown (train):")
print(sample_train_labels["label_source"].value_counts())

# ── Corrected time estimate ──────────────────────────────────────────────
n_debug_total     = transactions_debug["msno"].nunique()
n_full_total      = transactions["msno"].nunique()
scale             = n_full_total / max(n_debug_total, 1)
time_per_user     = (train_time + val_time) / max(n_debug_total, 1)
avg_pop           = (len(sample_train_labels) + len(sample_val_labels)) / 2
est_pop_full      = avg_pop * scale
est_per_month     = time_per_user * est_pop_full
n_months_total    = len(TRAIN_MONTHS) + len(VAL_MONTHS) + 1

print(f"\nCORRECTED TIME ESTIMATE (vectorized)")
print(f"  debug sample size         : {n_debug_total:,} users")
print(f"  debug time (2 months)     : {train_time + val_time:.2f}s")
print(f"  est. population/month     : ~{est_pop_full:,.0f} users")
print(f"  est. time/month (full)    : ~{est_per_month:.1f}s")
print(f"  est. total ({n_months_total} months)     : "
      f"~{est_per_month * n_months_total:.0f}s "
      f"(~{est_per_month * n_months_total / 60:.1f} min)")


Debug sample: 128,824 target-month users + 0 filler = 128,824 total
Debug transactions rows: 313,406

[2016-12] population=25,271  churn_rate=0.994  observed=21,955  no_future=3,316
[2017-01] population=24,749  churn_rate=0.975  observed=22,100  no_future=2,649
sample_train_labels shape : (25271, 5) (0.23s)
sample_val_labels shape   : (24749, 5) (0.27s)

Label source breakdown (train):
label_source
observed          21955
no_future_data     3316
Name: count, dtype: int64

CORRECTED TIME ESTIMATE (vectorized)
  debug sample size         : 128,824 users
  debug time (2 months)     : 0.51s
  est. population/month     : ~255,433 users
  est. time/month (full)    : ~1.0s
  est. total (3 months)     : ~3s (~0.1 min)



## 8A. Core feature builder adjusted

Feature builder still follows:
- all prior activity before cutoff
- restricted to `population_users`
- transactions + members only


In [77]:

# ===== 8A. build_core_feature_snapshot adjusted =====
def build_core_feature_snapshot(
    cutoff_date,
    transactions_df,
    members_df,
    population_users=None,
):
    cutoff_date = pd.to_datetime(cutoff_date)

    tx = transactions_df[pd.to_datetime(transactions_df["transaction_date"], errors="coerce") <= cutoff_date].copy()
    mem = members_df.copy()

    if population_users is not None:
        tx = tx[tx["msno"].astype("string").isin(population_users)].copy()
        mem = mem[mem["msno"].astype("string").isin(population_users)].copy()

    mem_feat = mem.copy()
    # members_core is already cleaned in preprocessing (bd, city, gender, registered_via)
    # No re-cleaning needed here — avoids double bd_missing flag creation

    # registration_init_time is dropped in preprocessing; use registration_year/month/day
    if "registration_year" in mem_feat.columns and "registration_month" in mem_feat.columns:
        reg_date = pd.to_datetime(
            mem_feat["registration_year"].astype(str) + "-" +
            mem_feat["registration_month"].astype(str).str.zfill(2) + "-01",
            errors="coerce"
        )
        mem_feat["days_since_reg"] = (cutoff_date - reg_date).dt.days.clip(lower=0)

    mem_keep = ["msno"] + [c for c in [
        "bd", "bd_missing",
        "city", "city_missing",
        "gender", "gender_missing",
        "registered_via", "registered_via_missing",
        "days_since_reg",
    ] if c in mem_feat.columns]
    mem_feat = mem_feat[mem_keep].copy()

    tx_feat = pd.DataFrame(columns=["msno"])
    if len(tx) > 0:
        tx = tx.sort_values(["msno", "transaction_date", "membership_expire_date"])
        grp = tx.groupby("msno", dropna=False)

        tx_feat = pd.DataFrame({"msno": list(grp.size().index)})
        tx_feat["n_txns"] = grp.size().values

        if "is_auto_renew" in tx.columns:
            tx_feat["auto_renew_rate"]    = grp["is_auto_renew"].mean().values
            # NEW: most recent auto_renew flag — current intent, stronger signal than average
            tx_feat["last_is_auto_renew"] = grp["is_auto_renew"].last().values

        if "payment_plan_days" in tx.columns:
            tx_feat["avg_plan_days"]    = grp["payment_plan_days"].mean().values
            tx_feat["plan_days_std"]    = grp["payment_plan_days"].std().fillna(0).values
            # NEW: did user ever change plan? plan switching signals dissatisfaction
            tx_feat["plan_change_flag"] = (grp["payment_plan_days"].nunique() > 1).astype("int8").values

        if "actual_amount_paid" in tx.columns:
            tx_feat["total_amount_paid"] = grp["actual_amount_paid"].sum().values
            tx_feat["avg_amount_paid"] = grp["actual_amount_paid"].mean().values
            tx_feat["zero_paid_rate"] = grp["actual_amount_paid"].apply(lambda s: (s == 0).mean()).values

        if all(c in tx.columns for c in ["plan_list_price", "actual_amount_paid"]):
            price_mean = grp["plan_list_price"].mean()
            paid_mean = grp["actual_amount_paid"].mean()
            tx_feat["avg_discount_rate"] = ((price_mean - paid_mean) / price_mean.replace(0, np.nan)).values
            tx_feat["avg_discount_rate"] = tx_feat["avg_discount_rate"].replace([np.inf, -np.inf], np.nan)

        tx_date_col = pd.to_datetime(tx["transaction_date"], errors="coerce")
        total_cnt = grp.size()
        
        tx_recent_30 = tx[tx_date_col > (cutoff_date - pd.Timedelta(days=30))]
        grp_recent_30 = tx_recent_30.groupby("msno").size()
        tx_feat["share_30d"] = (grp_recent_30 / total_cnt).reindex(total_cnt.index).fillna(0).values

        tx_recent_90 = tx[tx_date_col > (cutoff_date - pd.Timedelta(days=90))]
        grp_recent_90 = tx_recent_90.groupby("msno").size()
        tx_feat["share_90d"] = (grp_recent_90 / total_cnt).reindex(total_cnt.index).fillna(0).values

        last_txn = pd.to_datetime(grp["transaction_date"].max(), errors="coerce")
        first_txn = pd.to_datetime(grp["transaction_date"].min(), errors="coerce")
        tx_feat["days_since_last_txn"] = (cutoff_date - last_txn).dt.days.values
        tx_feat["tenure_days"] = (cutoff_date - first_txn).dt.days.values

        if "is_cancel" in tx.columns:
            tx_feat["cancel_rate"]  = grp["is_cancel"].mean().values
            # NEW: absolute cancellation count — rate alone misses heavy churners
            tx_feat["cancel_count"] = grp["is_cancel"].sum().values
        if "payment_method_id" in tx.columns:
            tx_feat["payment_method_nunique"] = grp["payment_method_id"].nunique().values

        # NEW: days from last transaction to effective membership expiry
        # Large positive = user transacted well before expiry (proactive)
        # Near zero / negative = user let membership lapse before acting (churn risk)
        if "membership_expire_date" in tx.columns:
            last_expire_per_user = pd.to_datetime(
                grp["membership_expire_date"].max(), errors="coerce"
            )
            tx_feat["days_last_txn_to_expire"] = (
                last_expire_per_user - last_txn
            ).dt.days.values

    user_pool = pd.Series(dtype="string")
    for part in [mem_feat, tx_feat]:
        if "msno" in part.columns:
            user_pool = pd.concat([user_pool, part["msno"].astype("string")], ignore_index=True)

    out = pd.DataFrame({"msno": user_pool.dropna().drop_duplicates()})
    out["snapshot_date"] = cutoff_date

    for part in [mem_feat, tx_feat]:
        if "msno" in part.columns:
            out = out.merge(part, on="msno", how="left")

    # ── Final type enforcement + NaN/inf cleanup ────────────────────────
    # Guarantees output parquet has zero NaN/inf in numeric cols.
    SKIP_COLS = {"msno", "snapshot_date"}
    for c in out.columns:
        if c in SKIP_COLS:
            continue
        dtype_str = str(out[c].dtype)
        if "string" in dtype_str.lower() or dtype_str in ["object", "category"]:
            out[c] = out[c].astype(object).fillna("Unknown").astype(str)
        else:
            out[c] = pd.to_numeric(out[c], errors="coerce")
            out[c] = out[c].replace([np.inf, -np.inf], np.nan).fillna(0)

    # Sanity assertion
    num_out = out.select_dtypes(include="number")
    nan_rem = num_out.isna().sum().sum()
    inf_rem = np.isinf(num_out).sum().sum()
    if nan_rem > 0 or inf_rem > 0:
        import warnings
        warnings.warn(f"[build_core_feature_snapshot] "
                      f"{nan_rem} NaN + {inf_rem} inf remain after cleanup.")

    return out
def build_core_feature_snapshot(
    cutoff_date,
    transactions_df,
    members_df,
    population_users=None,
):
    cutoff_date = pd.to_datetime(cutoff_date)

    tx = transactions_df[pd.to_datetime(transactions_df["transaction_date"], errors="coerce") <= cutoff_date].copy()
    mem = members_df.copy()

    if population_users is not None:
        tx = tx[tx["msno"].astype("string").isin(population_users)].copy()
        mem = mem[mem["msno"].astype("string").isin(population_users)].copy()

    mem_feat = mem.copy()
    # members_core is already cleaned in preprocessing (bd, city, gender, registered_via)
    # No re-cleaning needed here — avoids double bd_missing flag creation

    # registration_init_time is dropped in preprocessing; use registration_year/month/day
    if "registration_year" in mem_feat.columns and "registration_month" in mem_feat.columns:
        reg_date = pd.to_datetime(
            mem_feat["registration_year"].astype(str) + "-" +
            mem_feat["registration_month"].astype(str).str.zfill(2) + "-01",
            errors="coerce"
        )
        mem_feat["days_since_reg"] = (cutoff_date - reg_date).dt.days.clip(lower=0)

    mem_keep = ["msno"] + [c for c in [
        "bd", "bd_missing",
        "city", "city_missing",
        "gender", "gender_missing",
        "registered_via", "registered_via_missing",
        "days_since_reg",
    ] if c in mem_feat.columns]
    mem_feat = mem_feat[mem_keep].copy()

    tx_feat = pd.DataFrame(columns=["msno"])
    if len(tx) > 0:
        tx = tx.sort_values(["msno", "transaction_date", "membership_expire_date"])
        grp = tx.groupby("msno", dropna=False)

        tx_feat = pd.DataFrame({"msno": list(grp.size().index)})
        tx_feat["n_txns"] = grp.size().values

        if "is_auto_renew" in tx.columns:
            tx_feat["auto_renew_rate"]    = grp["is_auto_renew"].mean().values
            # NEW: most recent auto_renew flag — current intent, stronger signal than average
            tx_feat["last_is_auto_renew"] = grp["is_auto_renew"].last().values

        if "payment_plan_days" in tx.columns:
            tx_feat["avg_plan_days"]    = grp["payment_plan_days"].mean().values
            tx_feat["plan_days_std"]    = grp["payment_plan_days"].std().fillna(0).values
            # NEW: did user ever change plan? plan switching signals dissatisfaction
            tx_feat["plan_change_flag"] = (grp["payment_plan_days"].nunique() > 1).astype("int8").values

        if "actual_amount_paid" in tx.columns:
            tx_feat["total_amount_paid"] = grp["actual_amount_paid"].sum().values
            tx_feat["avg_amount_paid"] = grp["actual_amount_paid"].mean().values
            tx_feat["zero_paid_rate"] = grp["actual_amount_paid"].apply(lambda s: (s == 0).mean()).values

        if all(c in tx.columns for c in ["plan_list_price", "actual_amount_paid"]):
            price_mean = grp["plan_list_price"].mean()
            paid_mean = grp["actual_amount_paid"].mean()
            tx_feat["avg_discount_rate"] = ((price_mean - paid_mean) / price_mean.replace(0, np.nan)).values
            tx_feat["avg_discount_rate"] = tx_feat["avg_discount_rate"].replace([np.inf, -np.inf], np.nan)

        tx_date_col = pd.to_datetime(tx["transaction_date"], errors="coerce")
        total_cnt = grp.size()
        
        tx_recent_30 = tx[tx_date_col > (cutoff_date - pd.Timedelta(days=30))]
        grp_recent_30 = tx_recent_30.groupby("msno").size()
        tx_feat["share_30d"] = (grp_recent_30 / total_cnt).reindex(total_cnt.index).fillna(0).values

        tx_recent_90 = tx[tx_date_col > (cutoff_date - pd.Timedelta(days=90))]
        grp_recent_90 = tx_recent_90.groupby("msno").size()
        tx_feat["share_90d"] = (grp_recent_90 / total_cnt).reindex(total_cnt.index).fillna(0).values

        last_txn = pd.to_datetime(grp["transaction_date"].max(), errors="coerce")
        first_txn = pd.to_datetime(grp["transaction_date"].min(), errors="coerce")
        tx_feat["days_since_last_txn"] = (cutoff_date - last_txn).dt.days.values
        tx_feat["tenure_days"] = (cutoff_date - first_txn).dt.days.values

        if "is_cancel" in tx.columns:
            tx_feat["cancel_rate"]  = grp["is_cancel"].mean().values
            # NEW: absolute cancellation count — rate alone misses heavy churners
            tx_feat["cancel_count"] = grp["is_cancel"].sum().values
        if "payment_method_id" in tx.columns:
            tx_feat["payment_method_nunique"] = grp["payment_method_id"].nunique().values

        # NEW: days from last transaction to effective membership expiry
        # Large positive = user transacted well before expiry (proactive)
        # Near zero / negative = user let membership lapse before acting (churn risk)
        if "membership_expire_date" in tx.columns:
            last_expire_per_user = pd.to_datetime(
                grp["membership_expire_date"].max(), errors="coerce"
            )
            tx_feat["days_last_txn_to_expire"] = (
                last_expire_per_user - last_txn
            ).dt.days.values

    user_pool = pd.Series(dtype="string")
    for part in [mem_feat, tx_feat]:
        if "msno" in part.columns:
            user_pool = pd.concat([user_pool, part["msno"].astype("string")], ignore_index=True)

    out = pd.DataFrame({"msno": user_pool.dropna().drop_duplicates()})
    out["snapshot_date"] = cutoff_date

    for part in [mem_feat, tx_feat]:
        if "msno" in part.columns:
            out = out.merge(part, on="msno", how="left")

    for c in out.columns:
        if c in ["msno", "snapshot_date"]:
            continue
        if str(out[c].dtype) in ["object", "category", "string"]:
            out[c] = out[c].astype("string").fillna("Unknown")
        else:
            out[c] = out[c].replace([np.inf, -np.inf], np.nan).fillna(0)


    return out

## Bước 9 — Xây dựng Train và Validation (Bryan-style + Official Labels)

### Chiến lược hai lớp

```
                    ┌─────────────────────────────────────────────────┐
                    │  build_snapshot_with_official_labels(month)      │
                    └───────────────────┬─────────────────────────────┘
                                        │
             ┌──────────────────────────┴──────────────────────────┐
             │                                                      │
   train.csv / train_v2.csv                              Không có file
   available? → YES                                      → FALLBACK
             │                                                      │
   Population = users trong                    Population từ
   official label file                         build_labels_vectorized
   Labels = is_churn từ official                Labels từ FE pipeline
   label_source = "official"                    label_source = "fe_derived"
             │                                                      │
             └──────────────────────┬───────────────────────────────┘
                                    │
                    Features: LUÔN từ FE pipeline
                    build_core_feature_snapshot(cutoff, pop_users)
                    (full transaction history <= cutoff)
```

### Tại sao thiết kế này tốt?

| Thành phần | Nguồn | Lý do |
|-----------|-------|-------|
| **Labels** | Official `train.csv` / `train_v2.csv` | Ground-truth của competition, đã verified |
| **Features** | FE pipeline (transactions + members) | Toàn bộ lịch sử trước cutoff, no leakage |
| **Population** | Từ official file (nếu có) | Đảm bảo khớp với competition evaluation |

### Setup Bryan-style:
- **Train**: January 2017 — `train.csv` labels + features từ lịch sử ≤ 2017-01-31
- **Val**: February 2017 — `train_v2.csv` labels + features từ lịch sử ≤ 2017-02-28
- **Inf**: March 2017 — features only, không có labels (dùng để submit)

> **Fallback**: Nếu `train.csv` / `train_v2.csv` không có trong `Data/`,
> pipeline tự động dùng FE-derived labels và in cảnh báo.


In [78]:
# ===== 9. Build Train + Validation (Bryan-style + Official Labels) =====
# Strategy:
#   Train : Jan 2017 population — labels from official train.csv if available,
#           else fallback to FE-derived labels
#   Val   : Feb 2017 population — labels from official train_v2.csv if available,
#           else fallback to FE-derived labels
#   Features: always computed from our FE pipeline (full tx history <= cutoff)

def build_snapshot_with_official_labels(
    month: pd.Period,
    transactions: pd.DataFrame,
    members: pd.DataFrame,
    official_labels=None,
    grace_days: int = 30,
    split_name: str = "train",
):
    """
    Xây dựng snapshot cho một tháng:
    1. Luôn dùng FE pipeline để tính features (full history <= cutoff)
    2. Nếu có official_labels: dùng nhãn chính thức từ competition
       Nếu không: fallback sang FE-derived labels (build_labels_vectorized)
    3. Merge labels + features trên msno
    """
    cutoff = month_end(month)
    start  = time.time()

    if official_labels is not None:
        # ── Option B: official labels ──────────────────────────────────────
        # Population = users in official label file
        pop_users   = set(official_labels["msno"].astype(str).tolist())
        labels_snap = official_labels[["msno", "is_churn"]].copy()
        labels_snap["snapshot_date"] = cutoff
        labels_snap["last_expire"]   = pd.NaT      # not available from official file
        labels_snap["label_source"]  = "official"
        label_type = "official"
    else:
        # ── Fallback: FE-derived labels ────────────────────────────────────
        labels_snap = build_labels_vectorized(
            transactions, month, grace_days=grace_days, debug=False
        )
        if len(labels_snap) == 0:
            return None
        pop_users  = set(labels_snap["msno"].astype(str).tolist())
        label_type = "fe_derived"

    # ── Features: always from FE pipeline ──────────────────────────────────
    feat_snap = build_core_feature_snapshot(
        cutoff, transactions, members, population_users=pop_users
    )

    # ── Merge labels + features ────────────────────────────────────────────
    snap = labels_snap.merge(feat_snap, on=["msno", "snapshot_date"], how="left")
    snap["dataset_split"] = split_name

    elapsed    = time.time() - start
    churn_rate = float(snap["is_churn"].mean()) if "is_churn" in snap.columns else 0
    print(
        f"  [{split_name}] {month}: pop={len(pop_users):,} | "
        f"labels={len(labels_snap):,} | snap={snap.shape[0]:,} | "
        f"churn={churn_rate:.2%} | source={label_type} | {elapsed:.2f}s"
    )
    return snap


print("\n" + "="*90)
print("BUILDING TRAIN SNAPSHOT — January 2017")
print("="*90)

pipeline_start = time.time()

train_snap = build_snapshot_with_official_labels(
    TRAIN_MONTHS[0],
    transactions, members,
    official_labels=official_train_labels,  # None if not found → fallback
    grace_days=GRACE_DAYS,
    split_name="train",
)
if train_snap is None:
    raise ValueError("✗ ERROR: Train snapshot is empty.")
train_dataset = train_snap

print("\n" + "="*90)
print("BUILDING VALIDATION SNAPSHOT — February 2017")
print("="*90)

val_snap = build_snapshot_with_official_labels(
    VAL_MONTHS[0],
    transactions, members,
    official_labels=official_val_labels,    # None if not found → fallback
    grace_days=GRACE_DAYS,
    split_name="validation",
)
if val_snap is None:
    raise ValueError("✗ ERROR: Validation snapshot is empty.")
val_dataset = val_snap

master_multi  = pd.concat([train_dataset, val_dataset], axis=0, ignore_index=True)
combined_time = time.time() - pipeline_start
gc.collect()

print(f"\n{'='*90}")
print(f"✓ TRAIN+VAL COMPLETE: {master_multi.shape[0]:,} rows × {master_multi.shape[1]:,} cols")
print(f"  Time: {combined_time:.2f}s")
print(f"{'='*90}")
print("\nRows per snapshot:")
print(master_multi.groupby("snapshot_date").size())
print("\nChurn rate per snapshot:")
print(master_multi.groupby("snapshot_date")["is_churn"].mean())
print("\nRows by split:")
print(master_multi.groupby("dataset_split").size())
print("\nLabel source breakdown:")
print(master_multi.groupby(["dataset_split", "label_source"]).size())
print("\nFirst 5 rows:")
display(master_multi.head())



BUILDING TRAIN SNAPSHOT — January 2017
  [train] 2017-01: pop=992,931 | labels=992,931 | snap=992,931 | churn=6.39% | source=official | 18.57s

BUILDING VALIDATION SNAPSHOT — February 2017
  [validation] 2017-02: pop=970,960 | labels=970,960 | snap=970,960 | churn=8.99% | source=official | 20.13s

✓ TRAIN+VAL COMPLETE: 1,963,891 rows × 33 cols
  Time: 39.03s

Rows per snapshot:
snapshot_date
2017-01-31    992931
2017-02-28    970960
dtype: int64

Churn rate per snapshot:
snapshot_date
2017-01-31    0.063923
2017-02-28    0.089942
Name: is_churn, dtype: float64

Rows by split:
dataset_split
train         992931
validation    970960
dtype: int64

Label source breakdown:
dataset_split  label_source
train          official        992931
validation     official        970960
dtype: int64

First 5 rows:


,msno,is_churn,snapshot_date,last_expire,label_source,bd,bd_missing,city,city_missing,gender,gender_missing,registered_via,registered_via_missing,days_since_reg,n_txns,auto_renew_rate,last_is_auto_renew,avg_plan_days,plan_days_std,plan_change_flag,total_amount_paid,avg_amount_paid,zero_paid_rate,avg_discount_rate,share_30d,share_90d,days_since_last_txn,tenure_days,cancel_rate,cancel_count,payment_method_nunique,days_last_txn_to_expire,dataset_split
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,1,2017-01-31,NaT,official,36.0,0.0,18,0.0,female,0.0,9,0.0,4323.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,1,2017-01-31,NaT,official,38.0,0.0,10,0.0,male,0.0,9,0.0,4323.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,1,2017-01-31,NaT,official,27.0,0.0,11,0.0,female,0.0,9,0.0,4140.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,1,2017-01-31,NaT,official,23.0,0.0,13,0.0,female,0.0,9,0.0,4109.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,1,2017-01-31,NaT,official,27.0,0.0,3,0.0,male,0.0,9,0.0,4079.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train



## 10. Build March inference snapshot


In [79]:
# ===== 10. Build March inference snapshot =====
print("\n" + "="*90)
print("BUILDING MARCH INFERENCE SNAPSHOT")
print("="*90)

inf_start = time.time()

# Population: users whose effective state expires in 2017-03
# Use build_labels_vectorized with a dummy future window (no labels needed)
march_pop_df = build_labels_vectorized(
    transactions, INF_MONTH, grace_days=GRACE_DAYS, debug=True
)
march_population = set(march_pop_df["msno"].astype(str).tolist())
print(f"March population size: {len(march_population):,}")

inference_snapshot = build_core_feature_snapshot(
    inf_cutoff, transactions, members, population_users=march_population
)
# Add snapshot_date alignment
inference_snapshot["snapshot_date"] = inf_cutoff

inf_time = time.time() - inf_start
print(f"inference_snapshot shape  : {inference_snapshot.shape}")
print(f"inference unique users    : {inference_snapshot['msno'].nunique():,}")
print(f"inference built in        : {inf_time:.2f}s")
print("\nFirst 5 rows:")
display(inference_snapshot.head())



BUILDING MARCH INFERENCE SNAPSHOT
[2017-03] population=32,808  churn_rate=0.987  observed=420  no_future=32,388
March population size: 32,808
inference_snapshot shape  : (32808, 29)
inference unique users    : 32,808
inference built in        : 5.49s

First 5 rows:


,msno,snapshot_date,bd,bd_missing,city,city_missing,gender,gender_missing,registered_via,registered_via_missing,days_since_reg,n_txns,auto_renew_rate,last_is_auto_renew,avg_plan_days,plan_days_std,plan_change_flag,total_amount_paid,avg_amount_paid,zero_paid_rate,avg_discount_rate,share_30d,share_90d,days_since_last_txn,tenure_days,cancel_rate,cancel_count,payment_method_nunique,days_last_txn_to_expire
0,QzyX6ufzLhmGeiE0cyytk0lC13hQp45q4BOVUL+zpTI=,2017-03-31,29.0,0.0,13,0.0,Unknown,1.0,9,0.0,912.0,3,1.0,1,30.0,0.0,0,509,169.666667,0.0,0.0,0.666667,0.666667,14,349,0.333333,1,2,30
1,fkLgfIOX0bWM9/BQQChOCDzoos23szsckxPvxrBbtmY=,2017-03-31,20.0,0.0,4,0.0,female,0.0,9,0.0,912.0,2,0.0,0,30.0,0.0,0,298,149.000000,0.0,0.0,0.000000,0.500000,30,371,0.000000,0,1,30
2,etZ6WY4Qrn76v3JqLj0xpT1zA3yCoL35lti7cLc4tik=,2017-03-31,27.0,1.0,1,0.0,Unknown,1.0,4,0.0,89.0,1,0.0,0,30.0,0.0,0,149,149.000000,0.0,0.0,0.000000,1.000000,30,30,0.000000,0,1,30
3,nq+4KRKNWTQkH9VNArdNfhBNl70Vh01WEi/i9rPlxqU=,2017-03-31,27.0,1.0,1,0.0,Unknown,1.0,13,0.0,58.0,1,1.0,1,30.0,0.0,0,129,129.000000,0.0,0.0,0.000000,1.000000,30,30,0.000000,0,1,30
4,c8MWafMse+6+aWZwpBddB5CQ6dH2uncxqSJX/hoj8f0=,2017-03-31,25.0,0.0,13,0.0,male,0.0,9,0.0,3742.0,2,1.0,1,30.0,0.0,0,298,149.000000,0.0,0.0,0.500000,0.500000,11,101,0.500000,1,1,-1



## 11. Auxiliary logs branch


In [80]:

# ===== 11. Auxiliary logs info =====
if user_logs_hist is not None:
    print("Historical logs users:", user_logs_hist["msno"].nunique())
    print("Historical logs range:", user_logs_hist["date"].min(), "->", user_logs_hist["date"].max())
if user_logs_march is not None:
    print("March logs users:", user_logs_march["msno"].nunique())
    print("March logs range:", user_logs_march["date"].min(), "->", user_logs_march["date"].max())


Historical logs users: 22443
Historical logs range: 2015-01-01 00:00:00 -> 2017-02-28 00:00:00
March logs users: 316345
March logs range: 2017-03-01 00:00:00 -> 2017-03-31 00:00:00



## 12. Export


In [81]:

# ===== 12. Export =====
master_export = master_multi.copy()

for col in master_export.columns:
    dtype_str = str(master_export[col].dtype)
    if dtype_str in ["category", "object", "string"]:
        master_export[col] = master_export[col].astype("string")
    elif "datetime" in dtype_str:
        master_export[col] = pd.to_datetime(master_export[col], errors="coerce")
    elif dtype_str == "bool":
        master_export[col] = master_export[col].astype("int8")

sort_cols = [c for c in ["snapshot_date", "msno"] if c in master_export.columns]
if len(sort_cols) > 0:
    master_export = master_export.sort_values(sort_cols).reset_index(drop=True)

meta_cols = [c for c in ["msno", "snapshot_date", "last_expire",
             "is_churn", "label_source", "dataset_split"]
             if c in master_export.columns]
feature_cols = [c for c in master_export.columns if c not in meta_cols]

master_model_table = master_export[meta_cols + feature_cols].copy()

master_export.to_parquet(DATA_DIR / "final_df.parquet", index=False)
master_export.to_parquet(DATA_DIR / "master_table.parquet", index=False)
master_model_table.to_parquet(DATA_DIR / "master_model_table.parquet", index=False)
master_model_table.to_parquet(DATA_DIR / "processed_master.parquet", index=False)

snapshot_summary = (
    master_export.groupby(["dataset_split", "snapshot_date"])
    .agg(
        n_rows=("is_churn", "size"),
        churn_rate=("is_churn", "mean"),
        n_users=("msno", "nunique")
    )
    .reset_index()
)
snapshot_summary.to_csv(DATA_DIR / "snapshot_summary.csv", index=False)

run_metadata = {
    "n_rows": int(master_export.shape[0]),
    "n_cols": int(master_export.shape[1]),
    "n_snapshots": int(master_export["snapshot_date"].nunique()),
    "feature_count": int(len(feature_cols)),
    "feature_names": feature_cols,
    "setup": {
        "train_months": [str(m) for m in TRAIN_MONTHS],
        "validation_months": [str(m) for m in VAL_MONTHS],
        "inference_month": str(INF_MONTH),
        "grace_days": GRACE_DAYS,
        "population_rule": "effective-state inspired via Scala-aligned state selection",
        "label_rule": "no valid renewal within 30 days after expiry",
        "feature_rule": "all prior activity before month-end cutoff"
    },
    "core_model_uses_logs": False,
    "notes": [
        "v7: vectorized label builder, Scala-aligned effective state",
        "Bryan-style split: Train=Jan 2017, Val=Feb 2017, Inf=Mar 2017",
        "Option B: official train.csv/train_v2.csv labels used if available",
        "Features always from FE pipeline regardless of label source",
        "label_source=official|fe_derived|no_future_data|beyond_window"
    ]
}

with open(DATA_DIR / "feature_engineering_metadata_v7.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2, default=str)

inf_export = inference_snapshot.copy()
for col in inf_export.columns:
    if str(inf_export[col].dtype) in ["category", "object", "string"]:
        inf_export[col] = inf_export[col].astype("string")
inf_export.to_parquet(DATA_DIR / "inference_snapshot.parquet", index=False)

print("Saved:")
print(DATA_DIR / "final_df.parquet")
print(DATA_DIR / "master_table.parquet")
print(DATA_DIR / "master_model_table.parquet")
print(DATA_DIR / "processed_master.parquet")
print(DATA_DIR / "snapshot_summary.csv")
print(DATA_DIR / "feature_engineering_metadata_v7.json")
print(DATA_DIR / "inference_snapshot.parquet")


Saved:
Data\final_df.parquet
Data\master_table.parquet
Data\master_model_table.parquet
Data\processed_master.parquet
Data\snapshot_summary.csv
Data\feature_engineering_metadata_v7.json
Data\inference_snapshot.parquet


In [82]:

# ===== 13. Final sanity =====
print("master_export shape:", master_export.shape)
print("snapshot_date unique:", master_export["snapshot_date"].nunique())

print("\nRows per snapshot:")
display(master_export.groupby("snapshot_date").size())

print("\nChurn rate per snapshot:")
display(master_export.groupby("snapshot_date")["is_churn"].mean())

print("\nRows by split:")
display(master_export.groupby("dataset_split").size())

print("\nLabel source breakdown:")
display(master_export.groupby(["dataset_split", "label_source"]).size())

print("\nFeature columns:")
print(feature_cols)

# Verify no meta cols leaked into features
meta_cols_check = ["msno", "snapshot_date", "last_expire", "is_churn",
                   "label_source", "dataset_split"]
leaked = [c for c in meta_cols_check if c in feature_cols]
print(f"\nMeta cols leaked into features: {leaked if leaked else 'NONE ✅'}")
print(f"Total features: {len(feature_cols)}")

print("\nReminder:")
print("  - If label_source='official': labels from train.csv/train_v2.csv (ground truth)")
print("  - If label_source='fe_derived': labels from FE pipeline (fallback)")
print("  - Filter label_source != 'beyond_window' before training")
print("  - Exclude meta cols from feature matrix X")


master_export shape: (1963891, 33)
snapshot_date unique: 2

Rows per snapshot:


snapshot_date
2017-01-31    992931
2017-02-28    970960
dtype: int64


Churn rate per snapshot:


snapshot_date
2017-01-31    0.063923
2017-02-28    0.089942
Name: is_churn, dtype: float64


Rows by split:


dataset_split
train         992931
validation    970960
dtype: int64


Label source breakdown:


dataset_split  label_source
train          official        992931
validation     official        970960
dtype: int64


Feature columns:
['bd', 'bd_missing', 'city', 'city_missing', 'gender', 'gender_missing', 'registered_via', 'registered_via_missing', 'days_since_reg', 'n_txns', 'auto_renew_rate', 'last_is_auto_renew', 'avg_plan_days', 'plan_days_std', 'plan_change_flag', 'total_amount_paid', 'avg_amount_paid', 'zero_paid_rate', 'avg_discount_rate', 'share_30d', 'share_90d', 'days_since_last_txn', 'tenure_days', 'cancel_rate', 'cancel_count', 'payment_method_nunique', 'days_last_txn_to_expire']

Meta cols leaked into features: NONE ✅
Total features: 27

Reminder:
  - If label_source='official': labels from train.csv/train_v2.csv (ground truth)
  - If label_source='fe_derived': labels from FE pipeline (fallback)
  - Filter label_source != 'beyond_window' before training
  - Exclude meta cols from feature matrix X
